# 🚀 TabDiff Stage 1/8: Epochs 0 ➔ 1000
**Course Project:** Diffusion and Flow Matching Generative AI (DFMGAI)  
**Target Hardware:** Free Google Colab T4 GPU  
**Estimated Time:** ~15–20 minutes  
**Goal:** Train from Epoch 0 to 1000 and save `model_1000.pt`.


## 1. Verify GPU Availability


In [ ]:
!nvidia-smi


## 2. Mount Google Drive (Recommended for Zero-Friction Checkpointing)
Mounting Google Drive allows all 8 notebooks to automatically pass checkpoints to each other without manual upload/download.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive connected! Checkpoints will sync automatically.")
except Exception as e:
    print("ℹ️ Running without Google Drive. You can use manual file upload below.")


## 3. Clone / Update Repository


In [ ]:
import os
if not os.path.exists("/content/Project_tabdiff"):
    !git clone https://github.com/rider9600/Project_tabdiff.git /content/Project_tabdiff
%cd /content/Project_tabdiff
!git pull origin main
%cd /content/Project_tabdiff/baseline_tabdiff


## 4. Install Dependencies & Prepare Adult Dataset


In [ ]:
!pip install -q toml tomli tomli-w category_encoders scikit-learn pandas scipy xgboost lightgbm prdc sdmetrics rdt icecream ml_collections imbalanced-learn kaleido
!python download_dataset.py
!python process_dataset.py --dataname adult


## 5. Train Epochs 0 ➔ 1000
Runs for exactly 1,000 epochs (~15 minutes on T4 GPU) and saves `model_1000.pt`.


In [ ]:
!python main.py --dataname adult --method tabdiff --steps 1000 --check_val_every 1000 --no_wandb


## 6. Save Checkpoint `model_1000.pt` (Drive & PC Download)


In [ ]:
import os, shutil
from google.colab import files
ckpt_dir = "tabdiff/ckpt/adult/learnable_schedule"
gdrive_dir = "/content/drive/MyDrive/TabDiff_Checkpoints"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dir, exist_ok=True)
    if os.path.exists(f"{ckpt_dir}/model_1000.pt"):
        shutil.copy(f"{ckpt_dir}/model_1000.pt", f"{gdrive_dir}/model_1000.pt")
    if os.path.exists(f"{ckpt_dir}/ema_model_1000.pt"):
        shutil.copy(f"{ckpt_dir}/ema_model_1000.pt", f"{gdrive_dir}/ema_model_1000.pt")
    print("✅ Checkpoint backed up to Google Drive!")
zip_name = "tabdiff_stage_1_ckpt.zip"
!zip -j {zip_name} tabdiff/ckpt/adult/learnable_schedule/model_1000.pt tabdiff/ckpt/adult/learnable_schedule/ema_model_1000.pt 2>/dev/null || true
if os.path.exists(zip_name):
    files.download(zip_name)
    print(f"✅ Download prompt opened for {zip_name}")
